# Config 

> Configuration options and global defaults

In [ ]:
#| default_exp utils.config

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:

#| export
from __future__ import annotations
from typing import Dict, Optional, Any
import os, json
from pathlib import Path
from types import SimpleNamespace

## Config object
Dot-accessible, mutable config inspired by scanpy.cfg; remains optional thanks to helper functions.

In [ ]:
#| export

def env_int(name: str, default: int) -> int:
    """Return an int env value with a safe fallback."""
    try:
        return int(os.environ.get(name, default))
    except (TypeError, ValueError):
        return default


class DotConfig(dict):
    """Dot-accessible configuration with dictionary semantics.
    
    A dictionary subclass that allows attribute-style access to keys.
    Useful for configuration objects where config.key is more
    readable than config['key'].
    
    Examples
    --------
    >>> cfg = DotConfig({'debug': True})
    >>> cfg.debug
    True
    >>> cfg.new_key = 'value'
    >>> cfg['new_key']
    'value'
    """
    def __getattr__(self, key: str):
        try:
            return self[key]
        except KeyError as exc:
            raise AttributeError(key) from exc

    def __setattr__(self, key: str, value: Any) -> None:
        self[key] = value

    def copy(self) -> "DotConfig":
        """Return a shallow copy as another DotConfig.
        
        Returns
        -------
        DotConfig
            A new DotConfig instance with the same key-value pairs.
        """
        return DotConfig(self)

    def update_from_env(self) -> None:
        """Refresh config from environment variables if present.
        
        Updates log_level, prompt_dir, env_dir_name, and timeouts from
        NBMCP_LOG_LEVEL, NBMCP_PROMPT_DIR, NBMCP_ENV_DIR, and timeout
        env vars (NBMCP_TIMEOUT_*).
        """
        self.log_level = os.environ.get("NBMCP_LOG_LEVEL", self.log_level)
        self.prompt_dir = os.environ.get("NBMCP_PROMPT_DIR", self.prompt_dir)
        self.env_dir_name = os.environ.get("NBMCP_ENV_DIR", self.env_dir_name)
        self.timeout_analyze_remote = env_int("NBMCP_TIMEOUT_ANALYZE_REMOTE", self.timeout_analyze_remote)
        self.timeout_execute_cell = env_int("NBMCP_TIMEOUT_EXECUTE_CELL", self.timeout_execute_cell)
        self.timeout_discover_editables = env_int("NBMCP_TIMEOUT_DISCOVER_EDITABLES", self.timeout_discover_editables)
        self.timeout_run_tutorials = env_int("NBMCP_TIMEOUT_RUN_TUTORIALS", self.timeout_run_tutorials)


DEFAULT_CONFIG = DotConfig({
    "log_level": os.environ.get("NBMCP_LOG_LEVEL", "INFO"),
    "prompt_dir": os.environ.get("NBMCP_PROMPT_DIR", "prompt_templates"),
    "env_dir_name": os.environ.get("NBMCP_ENV_DIR", "Environments"),
    "max_tree_files": 600,
    "timeout_analyze_remote": env_int("NBMCP_TIMEOUT_ANALYZE_REMOTE", 60),
    "timeout_execute_cell": env_int("NBMCP_TIMEOUT_EXECUTE_CELL", 60),
    "timeout_discover_editables": env_int("NBMCP_TIMEOUT_DISCOVER_EDITABLES", 30),
    "timeout_run_tutorials": env_int("NBMCP_TIMEOUT_RUN_TUTORIALS", 600),
})
"""Default settings used by nbdev-mcp. Environment variables can override these defaults."""

NBMCP_CONFIG = DEFAULT_CONFIG
"""Singleton configuration shared across the package. Dot-accessible (NBMCP_CONFIG.log_level)."""

CURRENT_PROJECT: Optional[Path] = None
"""Currently selected nbdev project; None until explicitly set."""


In [ ]:
#| export
def configure(**kwargs: Any) -> DotConfig:
    '''Update the global NBMCP_CONFIG in-place and return it.
    
    Parameters
    ----------
    **kwargs : Any
        Key-value pairs to merge into the global config.
    
    Returns
    -------
    DotConfig
        The updated global configuration object.
    '''
    NBMCP_CONFIG.update(kwargs)
    return NBMCP_CONFIG


def get_config() -> DotConfig:
    '''Return the global config object (mutable, dot-accessible).
    
    Returns
    -------
    DotConfig
        The global NBMCP_CONFIG instance.
    '''
    return NBMCP_CONFIG


def get_current_project() -> Optional[Path]:
    '''Return the currently active project path (if any).
    
    Returns
    -------
    Path or None
        The current project path, or None if not set.
    '''
    return CURRENT_PROJECT


def set_current_project(path: Optional[Path]) -> None:
    '''Set the active project path used by tools/resources.
    
    Updates both config.CURRENT_PROJECT and paths.CURRENT_PROJECT
    to ensure consistent state across all modules.
    
    Parameters
    ----------
    path : Path or None
        The project path to set, or None to clear.
    '''
    global CURRENT_PROJECT
    CURRENT_PROJECT = path
    
    # Also update paths module to keep in sync
    # (paths imports CURRENT_PROJECT by value at load time)
    try:
        import nbdev_mcp.utils.paths as paths_module
        paths_module.CURRENT_PROJECT = path
    except ImportError:
        pass  # paths not loaded yet, will pick up from config


## Config & Bookmarks

In [ ]:
#| export
def find_config_dir() -> Path:
    '''Return the directory for storing MCP nbdev config (bookmarks).

    On macOS/Linux it defaults to ``~/.config/mcp.nbdev``; on Windows it uses
    ``%APPDATA%/mcp.nbdev``. The directory is created if missing.
    
    Returns
    -------
    Path
        The configuration directory path.
    '''
    if os.name == "nt":
        base = Path(os.environ.get("APPDATA", Path.home() / "AppData" / "Roaming"))
    else:
        base = Path(os.environ.get("XDG_CONFIG_HOME", Path.home() / ".config"))
    d = base / "mcp.nbdev"
    d.mkdir(parents=True, exist_ok=True)
    return d


BOOKMARKS_PATH: Path = find_config_dir() / "projects.json"
'''Path to the bookmarks JSON file for nbdev-mcp.'''


def load_bookmarks() -> Dict[str, str]:
    '''Load saved project aliases from BOOKMARKS_PATH.
    
    Returns
    -------
    Dict[str, str]
        Mapping of alias names to project paths.
    '''
    if BOOKMARKS_PATH.exists():
        try:
            data = json.loads(BOOKMARKS_PATH.read_text(encoding="utf-8"))
            aliases = data.get("aliases", {}) if isinstance(data, dict) else {}
            return {k: str(Path(v)) for k, v in aliases.items()}
        except Exception:
            return {}
    return {}


def save_bookmarks(aliases: Dict[str, str]) -> None:
    '''Persist project aliases to BOOKMARKS_PATH.
    
    Parameters
    ----------
    aliases : Dict[str, str]
        Mapping of alias names to project paths.
    '''
    BOOKMARKS_PATH.write_text(json.dumps({"aliases": aliases}, indent=2), encoding="utf-8")


## Expected Templates

In [ ]:
#| export


EXPECTED_PROMPT_TEMPLATES = (
    'workflow_philosophy.md',
    'nbdev_principles.md',
    'documentation_best_practices.md',
    'future_imports_guidance.md',
    'nbdev_howto.md',
    'documentation_guideline.md',
    'module_scaffold.md',
    'advanced_patterns.md',
    'main_pattern.md',
    'mcp_persistency.md',
    'reuse_first_checklist.md',
    'git_best_practices.md',
    'python312_features.md',
)
'''Tuple of expected prompt template filenames under `nbs/21_prompt_templates/` which get bundled with `nbdev_mcp`.'''

In [ ]:
print(*sorted(EXPECTED_PROMPT_TEMPLATES), sep='\n')

advanced_patterns.md
documentation_best_practices.md
documentation_guideline.md
future_imports_guidance.md
main_pattern.md
mcp_persistency.md
module_scaffold.md
nbdev_howto.md
nbdev_principles.md
reuse_first_checklist.md
workflow_philosophy.md


## Next

In [ ]:
#| export

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()